# Session 02 · Lab — Training a Variational Autoencoder on Fashion-MNIST

**Course:** Desarrollo y Evaluación de Modelos Propios · **Session 02:** Training Generative Models

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/02-training-generative-models/lab.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/02-training-generative-models/lab.ipynb)

In the theory block we followed the training loop (forward → loss → backward → step), the ELBO and the
reparameterization trick. Now you will build all of it in PyTorch: a small **variational autoencoder (VAE)**
that learns to compress images of clothes into 8 numbers and to draw new ones.

### Learning objectives
1. Turn an image dataset into normalized PyTorch tensors and data loaders.
2. Implement a VAE: encoder, reparameterization trick, decoder and the ELBO loss.
3. Write the training loop and read its loss curves (reconstruction vs KL).
4. Use the trained model to reconstruct, sample and interpolate in latent space.
5. Compare SGD and Adam on the same model and data.

### Time plan (60 min)
| Part | Topic | Time |
|---|---|---|
| 1 | Data: Fashion-MNIST → tensors in [0, 1] | 10' |
| 2 | The model: VAE, reparameterization and ELBO loss | 12' |
| 3 | The training loop and its loss curves | 13' |
| 4 | Generation: reconstructions, samples, interpolation, latent map | 15' |
| 5 | Optimizers: SGD vs Adam, one epoch each | 10' |

### ✅ Kaggle checklist
- **Settings → Accelerator → GPU T4 x2** (recommended). The lab also runs on CPU; training just takes a few minutes longer.
- **Settings → Internet → On** (the dataset is downloaded from the Hugging Face Hub).
- Run the cells **in order**, from top to bottom. If you change something in Part 2, re-run Parts 3–5.

In [ ]:
# --- Course setup (do not edit) ---
import os, sys, random, subprocess, importlib
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"  # used by the course smoke test
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

def ensure(package, import_name=None):
    """Install a package only if it is missing (Kaggle already ships most of them)."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # models and datasets come from the Hugging Face Hub
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "No internet connection: this notebook downloads models and datasets from Hugging Face.\n"
            "Kaggle: in the right-hand panel open Settings (Session options) and switch Internet on. "
            "The switch only appears once your account is phone-verified (kaggle.com/settings). "
            "Then run this cell again. Colab has internet by default."
        ) from None

import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Kaggle: {IN_KAGGLE} | Colab: {IN_COLAB} | Fast dev run: {FAST_DEV_RUN}")
# --- end of course setup ---
ensure("datasets")
ensure("matplotlib")

## Part 1 — Data: from images to tensors (10')

**Fashion-MNIST** (Xiao, Rasul & Vollgraf, 2017) contains 70,000 grayscale images of 28 × 28 pixels in
10 garment classes: 60,000 for training and 10,000 for testing. It was designed as a harder drop-in
replacement for the handwritten-digit dataset MNIST. We load it from the Hugging Face Hub.

To keep the lab fast we use a **random subset** of the training split, and part of the official test split as
our **validation** set (to watch for overfitting). An honest final *test* would need a third split that we never
look at while developing — remember the train / validation / test slide.

In [ ]:
import time
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset

# Sizes (the smoke test uses the small values; in class we use the large ones)
N_TRAIN = 1_000 if FAST_DEV_RUN else 20_000   # training images
N_VAL = 500 if FAST_DEV_RUN else 2_000        # validation images
EPOCHS = 1 if FAST_DEV_RUN else 10
BATCH_SIZE = 128
LATENT_DIM = 8

raw = load_dataset("zalando-datasets/fashion_mnist")
CLASS_NAMES = [name.replace(" - ", "-").replace(" / ", "/") for name in raw["train"].features["label"].names]
print(raw)
print("Classes:", CLASS_NAMES)

Each example is a PIL image plus an integer label. The network needs **float tensors**: we convert every
image to a NumPy array (values 0–255), divide by 255 so that pixels lie in **[0, 1]**, and flatten the 28 × 28
grid into a vector of **784** numbers, because our encoder is a fully connected (MLP) network.

In [ ]:
def to_tensors(split, n, seed=SEED):
    """Random subset of `split` → (float tensor N×784 with values in [0, 1], integer labels)."""
    subset = raw[split].shuffle(seed=seed).select(range(n))
    pixels = np.stack([np.asarray(img, dtype=np.float32) for img in subset["image"]])  # N × 28 × 28, 0–255
    x = torch.from_numpy(pixels / 255.0).reshape(n, -1)                               # N × 784, 0–1
    y = torch.tensor(subset["label"])
    return x, y

x_train, y_train = to_tensors("train", N_TRAIN)
x_val, y_val = to_tensors("test", N_VAL)
print("x_train:", tuple(x_train.shape), x_train.dtype, "| min", float(x_train.min()), "| max", float(x_train.max()))
print("x_val:  ", tuple(x_val.shape))
print("Images per class (train):", np.bincount(y_train.numpy(), minlength=10).tolist())

In [ ]:
def show_images(images, titles=None, ncols=10, size=1.25, suptitle=None):
    """Plot a batch of 784-pixel images (values in [0, 1]) as a grid."""
    images = torch.as_tensor(images).detach().float().cpu().reshape(-1, 28, 28)
    nrows = int(np.ceil(len(images) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows * (1.3 if titles else 1.0)))
    for i, ax in enumerate(np.array(axes).reshape(-1)):
        ax.axis("off")
        if i < len(images):
            ax.imshow(images[i], cmap="gray", vmin=0, vmax=1)
            if titles is not None:
                ax.set_title(titles[i], fontsize=8)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

first_of_each = [int((y_train == c).nonzero()[0]) for c in range(10)]
show_images(x_train[first_of_each], titles=CLASS_NAMES, suptitle="One training image per class")

Finally, **data loaders** serve the data in shuffled mini-batches of 128 images. A seeded generator makes the
shuffling order reproducible, which matters when we compare two runs (Part 5).

In [ ]:
loader_rng = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True, generator=loader_rng)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=512)
print(f"{len(train_loader)} training steps (mini-batches) per epoch")

### ✋ Checkpoint (answer in the Zoom chat)
1. Why do we divide the pixel values by 255?
2. Dividing by 255 happens *before* the train/validation split, yet it does not leak information. Why would a
   z-score scaler (subtract the mean, divide by the standard deviation) be different?
3. How many steps per epoch would we have with 60,000 training images and batches of 128?

### 🧪 Try it
Compute and plot the **average image** of the class `Trouser` in the training subset. What does it tell you?

<details><summary>Show a solution</summary>

```python
trouser = CLASS_NAMES.index("Trouser")
mean_img = x_train[y_train == trouser].mean(dim=0)
show_images(mean_img[None], titles=["Mean trouser"], ncols=1, size=2.5)
```
The mean image is a sharp, centred pair of trousers: Fashion-MNIST images are aligned and centred, which
is one reason a small MLP can model them. Answers to the checkpoint: (1) the BCE loss needs targets in
[0, 1] and small inputs help optimisation; (2) 255 is a fixed constant, while a z-score scaler *learns* a
mean and a standard deviation from the data, so it must be fitted on the training split only;
(3) 60,000 / 128 = 468.75, so 469 steps.
</details>

## Part 2 — The model: a variational autoencoder (12')

```
x (784) → [Linear 784→256, ReLU] → μ (8), log σ² (8) → z = μ + σ ⊙ ε → [Linear 8→256, ReLU, Linear 256→784] → logits (784)
```

- The **encoder** outputs the mean μ and the log-variance log σ² of a Gaussian q(z | x) for each image.
- The **reparameterization trick** draws z = μ + σ ⊙ ε with ε ~ N(0, I), so gradients can flow through μ and σ.
- The **decoder** turns z into 784 **logits**, one per pixel; `torch.sigmoid(logits)` gives pixel intensities in [0, 1].

The loss is the negative ELBO, per image and averaged over the batch:

$$\text{loss} = \underbrace{\sum_{\text{pixels}} \text{BCE}(x, \hat{x})}_{\text{reconstruction}} \; + \; \beta \cdot \underbrace{\left(-\tfrac{1}{2}\sum_{j=1}^{d} \big(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\big)\right)}_{\text{KL}\left(q(z|x)\,\|\,\mathcal{N}(0, I)\right)}$$

With β = 1 this is exactly the VAE of Kingma & Welling (2014). We use `binary_cross_entropy_with_logits`, which
combines the sigmoid and the BCE in one numerically stable operation.

In [ ]:
class VAE(nn.Module):
    """A small fully connected variational autoencoder for 28 × 28 images."""

    def __init__(self, latent_dim=8, hidden_dim=256, input_dim=784):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU())
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)       # mean of q(z|x)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)   # log-variance of q(z|x)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
                                     nn.Linear(hidden_dim, input_dim))  # one logit per pixel

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)   # σ = exp(½ · log σ²)
        eps = torch.randn_like(std)     # ε ~ N(0, I)
        return mu + std * eps           # z = μ + σ ⊙ ε

    def decode(self, z):
        return self.decoder(z)          # logits; torch.sigmoid(...) gives pixels in [0, 1]

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(logits, x, mu, logvar, beta=1.0):
    """Negative ELBO per image: BCE summed over pixels + beta * KL, both averaged over the batch."""
    batch_size = x.size(0)
    recon = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / batch_size
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
    return recon + beta * kl, recon, kl

Before training anything, a **sanity check**: run one batch through the untrained model and look at the shapes
and the loss.

In [ ]:
torch.manual_seed(SEED)
model = VAE(latent_dim=LATENT_DIM).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

x_batch, _ = next(iter(train_loader))
x_batch = x_batch.to(DEVICE)
with torch.no_grad():
    logits, mu, logvar = model(x_batch)
    loss, recon, kl = vae_loss(logits, x_batch, mu, logvar)
print("input", tuple(x_batch.shape), "| mu", tuple(mu.shape), "| logvar", tuple(logvar.shape), "| logits", tuple(logits.shape))
print(f"Untrained loss per image: {loss.item():.1f} = reconstruction {recon.item():.1f} + KL {kl.item():.3f}")
print(f"For reference, 784 × ln(2) = {784 * np.log(2):.1f}")

### ✋ Checkpoint
1. Why is the untrained reconstruction loss so close to 784 × ln 2 ≈ 543? *(Hint: what does the decoder predict for
   every pixel when its logits are close to 0?)*
2. Why is the KL almost 0 before training?

### 🧪 Try it
Compute the KL term by hand for a single latent dimension with μ = 1 and log σ² = 0. Then check it with `vae_loss`.

<details><summary>Show a solution</summary>

KL = −½ (1 + 0 − 1² − e⁰) = −½ (1 − 1 − 1) = **0.5**.

```python
mu_1 = torch.tensor([[1.0]]); logvar_1 = torch.tensor([[0.0]])
dummy_x = torch.zeros(1, 1); dummy_logits = torch.zeros(1, 1)
print(vae_loss(dummy_logits, dummy_x, mu_1, logvar_1)[2])   # tensor(0.5000)
```
Checkpoint answers: (1) logits near 0 mean a predicted probability of 0.5 for every pixel, and the BCE of 0.5
is ln 2 whatever the target; summed over 784 pixels that is 543.4. (2) A fresh encoder outputs μ ≈ 0 and
log σ² ≈ 0, which is already the prior N(0, I), so the KL is close to 0.
</details>

## Part 3 — The training loop and its loss curves (13')

These two functions are the heart of every training script you will write in this course. Compare them with the
slide *“Five lines you will write in every project”*: `zero_grad → forward → loss → backward → step`, with
`model.train()` for training and `model.eval()` plus `torch.no_grad()` for evaluation.

In [ ]:
def train_one_epoch(model, loader, optimizer, beta=1.0, step_log=None):
    """One pass over `loader`. Returns the average loss, reconstruction and KL per image."""
    model.train()
    totals, n_images = np.zeros(3), 0
    for x, _ in loader:
        x = x.to(DEVICE)
        optimizer.zero_grad()                         # 1 · clear old gradients
        logits, mu, logvar = model(x)                 # 2 · forward pass
        loss, recon, kl = vae_loss(logits, x, mu, logvar, beta)   # 3 · loss
        loss.backward()                               # 4 · gradients
        optimizer.step()                              # 5 · update the weights
        if step_log is not None:
            step_log.append(loss.item())
        totals += np.array([loss.item(), recon.item(), kl.item()]) * x.size(0)
        n_images += x.size(0)
    return dict(zip(["loss", "recon", "kl"], totals / n_images))


@torch.no_grad()
def evaluate(model, loader, beta=1.0):
    """Average loss, reconstruction and KL per image on `loader`, without updating the model."""
    model.eval()
    totals, n_images = np.zeros(3), 0
    for x, _ in loader:
        x = x.to(DEVICE)
        logits, mu, logvar = model(x)
        loss, recon, kl = vae_loss(logits, x, mu, logvar, beta)
        totals += np.array([loss.item(), recon.item(), kl.item()]) * x.size(0)
        n_images += x.size(0)
    return dict(zip(["loss", "recon", "kl"], totals / n_images))

In [ ]:
torch.manual_seed(SEED)
model = VAE(latent_dim=LATENT_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = {"epoch": [], "train_loss": [], "val_loss": [], "val_recon": [], "val_kl": []}
step_losses = []
start = time.time()
for epoch in range(1, EPOCHS + 1):
    train_stats = train_one_epoch(model, train_loader, optimizer, beta=1.0, step_log=step_losses)
    val_stats = evaluate(model, val_loader, beta=1.0)
    history["epoch"].append(epoch)
    history["train_loss"].append(train_stats["loss"])
    history["val_loss"].append(val_stats["loss"])
    history["val_recon"].append(val_stats["recon"])
    history["val_kl"].append(val_stats["kl"])
    print(f"Epoch {epoch:2d} | train loss {train_stats['loss']:6.1f} | val loss {val_stats['loss']:6.1f} "
          f"= recon {val_stats['recon']:6.1f} + KL {val_stats['kl']:5.2f} | {time.time() - start:4.0f} s")

In [ ]:
steps_per_epoch = len(train_loader)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(np.arange(1, len(step_losses) + 1) / steps_per_epoch, step_losses, lw=1, label="train loss (per step)")
axes[0].plot(history["epoch"], history["train_loss"], "o-", label="train loss (epoch mean)")
axes[0].plot(history["epoch"], history["val_loss"], "s--", label="validation loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss per image (−ELBO)"); axes[0].set_title("Training curves")
axes[0].set_ylim(top=min(max(step_losses), 600)); axes[0].legend()

axes[1].plot(history["epoch"], history["val_recon"], "o-", color="tab:blue", label="reconstruction")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("reconstruction (BCE)", color="tab:blue")
twin = axes[1].twinx()
twin.plot(history["epoch"], history["val_kl"], "s--", color="tab:red", label="KL")
twin.set_ylabel("KL", color="tab:red")
axes[1].set_title("Validation: the two terms of the loss")
plt.tight_layout()
plt.show()

### ✋ Checkpoint
1. Which term of the loss falls the most during training? What happens to the KL term?
2. Compare the last training and validation losses. Is there overfitting after 10 epochs?
3. The per-step curve is noisy while the epoch means are smooth. Where does the noise come from?

### 🧪 Try it
Train a second model with **β = 4** for 3 epochs and compare its validation reconstruction and KL with the model
above. Which term went up, and which went down?

<details><summary>Show a solution</summary>

```python
torch.manual_seed(SEED)
model_b4 = VAE(latent_dim=LATENT_DIM).to(DEVICE)
opt_b4 = torch.optim.Adam(model_b4.parameters(), lr=1e-3)
for epoch in range(1 if FAST_DEV_RUN else 3):
    train_one_epoch(model_b4, train_loader, opt_b4, beta=4.0)
print("beta=4:", evaluate(model_b4, val_loader, beta=1.0))
print("beta=1:", evaluate(model, val_loader, beta=1.0))
```
With a larger β the KL term goes **down** (codes stay closer to N(0, I)) and the reconstruction loss goes **up**
(blurrier images). We evaluate both models with β = 1 so that the totals are comparable (−ELBO).
Checkpoint answers: (1) reconstruction falls from about 543 to a much lower value; the KL rises from almost 0
in the first steps, because the encoder starts to *use* the latent code, and then stabilises. (2) Usually train and
validation stay close with this small model: little or no overfitting. (3) Each step uses a different random
mini-batch, so each loss is a noisy estimate.
</details>

## Part 4 — Generation: reconstruct, sample, interpolate (15')

### 4a · Reconstructions
We encode validation images and decode them again. For a clean picture we decode the **mean** code μ (no sampling).

In [ ]:
@torch.no_grad()
def reconstruct(model, x):
    model.eval()
    mu, _ = model.encode(x.to(DEVICE))
    return torch.sigmoid(model.decode(mu)).cpu()

one_per_class = [int((y_val == c).nonzero()[0]) for c in range(10)]
originals = x_val[one_per_class]
show_images(torch.cat([originals, reconstruct(model, originals)]), ncols=10,
            suptitle="Top: validation images · Bottom: reconstructions")

### 4b · New garments: sampling from the prior
This is what makes a VAE **generative**: draw z ~ N(0, I) and decode it. No input image is involved.

In [ ]:
@torch.no_grad()
def sample(model, n=20, seed=SEED):
    model.eval()
    g = torch.Generator().manual_seed(seed)
    z = torch.randn(n, model.latent_dim, generator=g).to(DEVICE)
    return torch.sigmoid(model.decode(z)).cpu()

show_images(sample(model, n=20), ncols=10, suptitle="20 samples decoded from z ~ N(0, I)")

### 4c · Latent interpolation
Encode two items, walk along the straight line between their codes, and decode each point. In a well-trained VAE
the path passes through plausible images, because the KL term filled the gaps in latent space.

In [ ]:
@torch.no_grad()
def interpolate(model, x_a, x_b, n_steps=10):
    model.eval()
    mu_a, _ = model.encode(x_a[None].to(DEVICE))
    mu_b, _ = model.encode(x_b[None].to(DEVICE))
    ts = torch.linspace(0, 1, n_steps, device=DEVICE)[:, None]
    z = (1 - ts) * mu_a + ts * mu_b
    return torch.sigmoid(model.decode(z)).cpu()

def first_of(label_name):
    return x_val[int((y_val == CLASS_NAMES.index(label_name)).nonzero()[0])]

show_images(interpolate(model, first_of("Sneaker"), first_of("Ankle boot")), ncols=10,
            suptitle="Sneaker → Ankle boot")
show_images(interpolate(model, first_of("Trouser"), first_of("Dress")), ncols=10,
            suptitle="Trouser → Dress")

### 4d · A map of the latent space
The latent space has 8 dimensions. To look at it we project the codes μ of the validation images onto their two
main directions (PCA), and colour each point by its class. We also count the **active dimensions**: those whose
μ actually varies across images (variance > 0.01, the criterion of Burda, Grosse & Salakhutdinov, 2016).
An inactive dimension ignores the input and just copies the prior. You will use this number in the challenge.

In [ ]:
@torch.no_grad()
def latent_means(model, x):
    model.eval()
    mu, _ = model.encode(x.to(DEVICE))
    return mu.cpu().numpy()

mus = latent_means(model, x_val)
centered = mus - mus.mean(axis=0)
_, _, vt = np.linalg.svd(centered, full_matrices=False)
coords = centered @ vt[:2].T

plt.figure(figsize=(7, 5.5))
for c in range(10):
    pts = coords[y_val.numpy() == c]
    plt.scatter(pts[:, 0], pts[:, 1], s=6, alpha=0.6, label=CLASS_NAMES[c])
plt.xlabel("principal direction 1"); plt.ylabel("principal direction 2")
plt.title("Validation images in latent space (PCA of μ)")
plt.legend(markerscale=3, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

active = int((mus.var(axis=0) > 0.01).sum())
print(f"Active latent dimensions: {active} of {LATENT_DIM}")
print("Variance of μ per dimension:", [round(float(v), 3) for v in mus.var(axis=0)])

### ✋ Checkpoint
1. Why do the samples in 4b look blurrier than the reconstructions in 4a?
2. Which classes overlap in the latent map? Does that match what you see in the images?
3. A retailer wants to “blend” two best-selling products into a new design idea. Which of the four outputs above
   does that, and what would you check before showing the result to a designer?

### 🧪 Try it
Interpolate between two items of your choice (for example `Bag` → `Sandal` or `Shirt` → `Coat`) using 12 steps.

<details><summary>Show a solution</summary>

```python
show_images(interpolate(model, first_of("Bag"), first_of("Sandal"), n_steps=12), ncols=12,
            suptitle="Bag → Sandal")
```
Checkpoint answers: (1) reconstructions start from a real image's code; prior samples may land in regions the
decoder rarely saw, and the model averages over uncertainty, so details blur. (2) Typically the upper-body
classes (T-shirt/top, Pullover, Coat, Shirt) overlap, and the footwear classes cluster together; they also look
alike in the images. (3) Latent interpolation; check that the intermediate images are plausible, that no training
image is copied, and that the source images may be used for this purpose.
</details>

## Part 5 — Optimizers: SGD vs Adam (10')

Same model, same initial weights, same mini-batches in the same order: only the optimizer changes. We train
**one epoch** with each and record the loss at every step.

In [ ]:
def one_epoch_run(make_optimizer):
    """Fresh VAE with the same initial weights and batch order; one epoch with the given optimizer."""
    torch.manual_seed(SEED)
    run_model = VAE(latent_dim=LATENT_DIM).to(DEVICE)
    optimizer = make_optimizer(run_model.parameters())
    loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    steps = []
    train_one_epoch(run_model, loader, optimizer, beta=1.0, step_log=steps)
    return steps, evaluate(run_model, val_loader, beta=1.0)

OPTIMIZERS = {
    "SGD (lr=1e-3)": lambda params: torch.optim.SGD(params, lr=1e-3),
    "SGD + momentum 0.9 (lr=1e-3)": lambda params: torch.optim.SGD(params, lr=1e-3, momentum=0.9),
    "Adam (lr=1e-3)": lambda params: torch.optim.Adam(params, lr=1e-3),
}
runs = {name: one_epoch_run(make) for name, make in OPTIMIZERS.items()}

plt.figure(figsize=(9, 4))
for name, (steps, _) in runs.items():
    plt.plot(steps, lw=1.2, label=name)
plt.xlabel("training step"); plt.ylabel("train loss per image"); plt.ylim(top=600)
plt.title("One epoch, three optimizers"); plt.legend()
plt.tight_layout()
plt.show()

print(f"{'Optimizer':32s} {'val loss':>9s} {'recon':>8s} {'KL':>7s}")
for name, (_, val) in runs.items():
    print(f"{name:32s} {val['loss']:9.1f} {val['recon']:8.1f} {val['kl']:7.2f}")

### ✋ Checkpoint
1. Which optimizer reached the lowest loss after one epoch? Connect it to the slide on Adam's per-weight step sizes.
2. Does “fastest after one epoch” guarantee “best after 50 epochs”? What would you need to check?

### 🧪 Try it
Plain SGD is often run with a larger learning rate. Add `"SGD (lr=1e-2)"` and `"SGD (lr=1e-4)"` to `OPTIMIZERS`
and re-run the cell. Which one is too small? Does the larger one catch up with Adam?

<details><summary>Show a solution</summary>

```python
OPTIMIZERS["SGD (lr=1e-2)"] = lambda params: torch.optim.SGD(params, lr=1e-2)
OPTIMIZERS["SGD (lr=1e-4)"] = lambda params: torch.optim.SGD(params, lr=1e-4)
runs = {name: one_epoch_run(make) for name, make in OPTIMIZERS.items()}
```
With lr = 1e-4 the curve barely moves in one epoch: the learning rate is too small. A larger SGD learning rate
speeds things up but can make the curve noisier; compare the final validation losses rather than your impression
of the curve. Checkpoint answers: (1) usually Adam, because it adapts the step size of each weight;
(2) no: compare validation curves over the whole training budget, and ideally several seeds.
</details>

## Wrap-up

**Key takeaways**
- The training loop is always the same five steps: `zero_grad → forward → loss → backward → step`, with
  `model.train()` for training and `model.eval()` + `torch.no_grad()` for evaluation.
- A VAE's loss has two terms that pull in opposite directions: **reconstruction** (sharp copies) and **KL**
  (a smooth latent space you can sample from). β sets the balance.
- Sampling z ~ N(0, I) generates new items; interpolating between codes blends two items.
- The optimizer and learning rate change how fast you get there: compare them on the same data, seed and budget.

**Next: the Session Challenge.** You will train your own VAE with an assigned latent size, β, learning rate and
three garment classes, compare it with a baseline (latent 8, β = 1), advise a fictional retailer, and debug a
training loop written by an AI assistant. Open `challenge.ipynb` from the session guide.

**References:** Kingma & Welling (2014, 2019) · Higgins et al. (2017) · Kingma & Ba (2015) ·
Xiao, Rasul & Vollgraf (2017) · Burda, Grosse & Salakhutdinov (2016) · PyTorch “Learn the Basics” tutorial.